# Notebook 02 · Una Deep Q-Network (DQN): la red sustituye a la tabla

**Introducción al Deep Learning — Módulo 4, Unidad 2: Aprendizaje por refuerzo (Parte II)**

En el Notebook 01 el agente aprendió una **tabla Q**. Funcionó porque solo había 16 estados.
Cuando los estados son muchos (los píxeles de una pantalla, la posición continua de un robot…),
la tabla **no cabe en memoria**.

La solución de DeepMind (Atari, 2015) fue el **DQN**: sustituir la tabla por una **red neuronal**
que recibe el estado y devuelve un valor Q por cada acción.

Aquí montaremos un DQN completo, con sus dos trucos clave:

- **Repetición de experiencia**: guardamos transiciones `(s, a, r, s')` y entrenamos con lotes al azar.
- **Red objetivo**: una copia congelada de la red para calcular el objetivo de forma estable.

Usaremos la **misma rejilla** para poder comparar con el Notebook 01, pero ahora el estado entra en la
red como un vector, no como un índice de tabla.

> 💡 En Colab, TensorFlow ya viene instalado. Entrenar tarda un poco (unos segundos por bloque de episodios).

## 1. Preparación y entorno

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np, matplotlib.pyplot as plt
from collections import deque
import random
tf.random.set_seed(0); np.random.seed(0); random.seed(0)

FILAS, COLS = 4, 4
META = (3, 3)
TRAMPAS = [(1, 1), (2, 3)]
N_ACCIONES = 4
MOV = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
FLECHAS = {0: "^", 1: "v", 2: "<", 3: ">"}

def paso(s, a):
    df, dc = MOV[a]
    f = min(max(s[0] + df, 0), FILAS - 1)
    c = min(max(s[1] + dc, 0), COLS - 1)
    ns = (f, c)
    if ns == META:    return ns, 10.0, True
    if ns in TRAMPAS: return ns, -10.0, True
    return ns, -1.0, False

print("Entorno listo.")

## 2. El estado como vector

La red no entiende de "casillas": necesita números. Codificamos cada estado como un vector **one-hot**
de 16 posiciones (un 1 en la casilla actual). Así estados distintos entran como vectores distintos.

In [ ]:
def a_vector(s):
    v = np.zeros(FILAS * COLS, dtype="float32")
    v[s[0] * COLS + s[1]] = 1.0
    return v

print("Estado (0,0) ->", a_vector((0, 0)))
print("Estado (1,2) ->", a_vector((1, 2)))

## 3. La red Q y la red objetivo

Una red pequeña: entrada de 16 (el estado) → capas ocultas → **4 salidas** (un valor Q por acción).

Creamos **dos** redes idénticas: la que entrenamos (`q_red`) y la **objetivo** (`q_target`), congelada,
que usamos para calcular los objetivos. Cada cierto tiempo copiamos los pesos de una a otra.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input

def crear_red():
    return Sequential([
        Input(shape=(FILAS * COLS,)),
        Dense(64, activation="relu"),
        Dense(64, activation="relu"),
        Dense(N_ACCIONES),              # salida lineal: un Q por accion
    ])

q_red = crear_red()
q_target = crear_red()
q_target.set_weights(q_red.get_weights())     # empiezan iguales
optimizador = tf.keras.optimizers.Adam(1e-3)
mse = tf.keras.losses.MeanSquaredError()
q_red.summary()

## 4. La memoria de repetición

Una cola donde guardamos las últimas transiciones `(s, a, r, s', fin)`. Para entrenar tomamos un
**lote aleatorio** de ella: así rompemos la correlación entre pasos consecutivos.

In [ ]:
memoria = deque(maxlen=5000)

def recordar(s, a, r, ns, fin):
    memoria.append((a_vector(s), a, r, a_vector(ns), fin))

## 5. El paso de entrenamiento

Con un lote de la memoria calculamos el **objetivo** de cada transición con la red objetivo:

$$\text{objetivo} = r + \gamma \max_{a'} Q_{\text{target}}(s', a') \quad(\text{y }=r\text{ si el estado es final})$$

y ajustamos `q_red` para que su predicción se acerque a ese objetivo (error cuadrático).

In [ ]:
GAMMA, BATCH = 0.9, 64

@tf.function
def entrenar_lote(estados, acciones, objetivos):
    with tf.GradientTape() as cinta:
        q = q_red(estados, training=True)
        idx = tf.stack([tf.range(tf.shape(acciones)[0]), acciones], axis=1)
        q_elegido = tf.gather_nd(q, idx)                 # Q de la accion tomada
        loss = mse(objetivos, q_elegido)
    grads = cinta.gradient(loss, q_red.trainable_variables)
    optimizador.apply_gradients(zip(grads, q_red.trainable_variables))
    return loss

def repetir_experiencia():
    if len(memoria) < BATCH:
        return
    lote = random.sample(memoria, BATCH)
    s, a, r, ns, fin = map(np.array, zip(*lote))
    q_next = q_target.predict(ns, verbose=0).max(axis=1)
    objetivos = r + GAMMA * q_next * (1 - fin.astype("float32"))
    entrenar_lote(tf.constant(s, dtype=tf.float32),
                  tf.constant(a, dtype=tf.int32),
                  tf.constant(objetivos, dtype=tf.float32))

## 6. El bucle principal (ε-greedy + DQN)

In [ ]:
def elegir(s, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(N_ACCIONES)
    q = q_red.predict(a_vector(s)[None, :], verbose=0)[0]
    return int(np.argmax(q))

EPISODIOS = 120
eps_ini, eps_fin = 1.0, 0.05
recompensas = []

for ep in range(EPISODIOS):
    epsilon = max(eps_fin, eps_ini * (1 - ep / EPISODIOS))
    s, total = (0, 0), 0.0
    for _ in range(50):
        a = elegir(s, epsilon)
        ns, r, fin = paso(s, a)
        recordar(s, a, r, ns, fin)
        repetir_experiencia()
        total += r; s = ns
        if fin: break
    if ep % 10 == 0:
        q_target.set_weights(q_red.get_weights())       # actualizar red objetivo
    recompensas.append(total)
    if ep % 20 == 0:
        print(f"episodio {ep:3d} | epsilon {epsilon:.2f} | recompensa {total:+.0f}")

print("Entrenamiento terminado.")

### La curva de aprendizaje del DQN

In [ ]:
def media_movil(x, n=10):
    return np.convolve(x, np.ones(n) / n, mode="valid")

plt.figure(figsize=(7, 3.2))
plt.plot(media_movil(recompensas), color="#FF647E")
plt.axhline(0, color="grey", linewidth=0.8)
plt.title("Recompensa por episodio (media móvil): el DQN aprende")
plt.xlabel("episodio"); plt.ylabel("recompensa"); plt.show()

## 7. La política que ha aprendido la red

Ahora no hay tabla: le pedimos a la red los Q de cada casilla y nos quedamos con el mejor.

In [ ]:
todos = np.array([a_vector((f, c)) for f in range(FILAS) for c in range(COLS)])
Q_red = q_red.predict(todos, verbose=0).reshape(FILAS, COLS, N_ACCIONES)
V = Q_red.max(axis=2)
politica = Q_red.argmax(axis=2)

fig, ax = plt.subplots(figsize=(4.4, 4.4))
ax.imshow(V, cmap="RdYlGn")
for f in range(FILAS):
    for c in range(COLS):
        if (f, c) == META:      ax.text(c, f, "META", ha="center", va="center", fontsize=8)
        elif (f, c) in TRAMPAS: ax.text(c, f, "X", ha="center", va="center", fontsize=14)
        else:                   ax.text(c, f, FLECHAS[politica[f, c]], ha="center", va="center", fontsize=20)
ax.set_xticks(range(COLS)); ax.set_yticks(range(FILAS))
ax.set_title("Política aprendida por la RED (no hay tabla)")
plt.show()

In [ ]:
def jugar():
    s, total = (0, 0), 0.0
    for _ in range(50):
        a = int(np.argmax(q_red.predict(a_vector(s)[None, :], verbose=0)[0]))
        s, r, fin = paso(s, a)
        total += r
        if fin: break
    return total

print("Recompensa media (política de la red, 20 partidas):",
      np.mean([jugar() for _ in range(20)]))

## Experimenta tú

1. Sube `EPISODIOS` a 300 y observa si la política se afina.
2. Cambia la frecuencia de actualización de la red objetivo (el `ep % 10`). ¿Qué pasa con `% 1`? ¿Y con `% 50`?
3. Quita la memoria de repetición (entrena solo con la última transición). ¿Se vuelve inestable?
4. Prueba una red más pequeña (una sola capa de 16). ¿Sigue aprendiendo?

## Resumen

- Un **DQN** sustituye la tabla Q por una **red neuronal** que mapea estado → Q de cada acción.
- **Generaliza**: no necesita haber visto un estado exacto para estimar su valor.
- **Repetición de experiencia**: entrenar con lotes aleatorios de transiciones pasadas da estabilidad.
- **Red objetivo**: una copia congelada evita que el objetivo se mueva mientras entrenamos.
- Es la técnica que permitió a las máquinas aprender a jugar a videojuegos directamente desde los píxeles.

Con esto cerramos la unidad de **aprendizaje por refuerzo**. En la siguiente clase pasaremos al
**despliegue de modelos** (M4 U3): cómo llevar un modelo entrenado a producción.